In [ ]:
import os
import cv2
import numpy as np

#  ABSOLUTE PROJECT PATH (NO RELATIVE PATHS)
PROJECT_ROOT = "/home/ryu/code/DL_project"

CNN_DIR = os.path.join(PROJECT_ROOT, "outputs/gradcam/cnn_grad")
MOBILENET_DIR = os.path.join(PROJECT_ROOT, "outputs/gradcam/mobilenet_grad")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs/gradcam/comparison")

print("CNN_DIR:", CNN_DIR)
print("MOBILENET_DIR:", MOBILENET_DIR)
print("OUT_DIR:", OUT_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

cnn_images = os.listdir(CNN_DIR)

for cnn_img in cnn_images:
    # Match filenames
    base_name = cnn_img.replace("cnn_grad_", "")
    mobilenet_img = f"mobilenet_grad_{base_name}"

    cnn_path = os.path.join(CNN_DIR, cnn_img)
    mobilenet_path = os.path.join(MOBILENET_DIR, mobilenet_img)

    if not os.path.exists(mobilenet_path):
        print("Skipping (no match):", base_name)
        continue

    cnn_img_data = cv2.imread(cnn_path)
    mobilenet_img_data = cv2.imread(mobilenet_path)

    if cnn_img_data is None or mobilenet_img_data is None:
        print("Skipping (read error):", base_name)
        continue

    # Resize to same height
    h = min(cnn_img_data.shape[0], mobilenet_img_data.shape[0])
    cnn_img_data = cv2.resize(cnn_img_data, (cnn_img_data.shape[1], h))
    mobilenet_img_data = cv2.resize(mobilenet_img_data, (mobilenet_img_data.shape[1], h))

    # Side-by-side
    combined = np.hstack((cnn_img_data, mobilenet_img_data))

    out_path = os.path.join(OUT_DIR, base_name)
    cv2.imwrite(out_path, combined)

    print("Saved comparison:", out_path)


CNN_DIR: /home/ryu/code/DL_project/outputs/gradcam/cnn_grad
MOBILENET_DIR: /home/ryu/code/DL_project/outputs/gradcam/mobilenet_grad
OUT_DIR: /home/ryu/code/DL_project/outputs/gradcam/comparison
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (223).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (191).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (84).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (281).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (14).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (129).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (183).jpg
Saved comparison: /home/ryu/code/DL_project/outputs/gradcam/comparison/BT-MRI Test GL (71).jpg
Saved comparison: /home/ryu/code/DL_proje

##### Quantitative comparision

In [ ]:


CNN_DIR = "/home/ryu/code/DL_project/outputs/gradcam/cnn_grad"
MOBILENET_DIR = "/home/ryu/code/DL_project/outputs/gradcam/mobilenet_grad"


def focus_score(heatmap, threshold=0.6):
    """
    Measures how focused a Grad-CAM heatmap is.
    Lower score = more localized attention.
    """
    return np.mean(heatmap > threshold)


def load_heatmap_from_gradcam(img_path):
    """
    Extract heatmap approximation from colored Grad-CAM image.
    Uses red channel as proxy.
    """
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap = img[:, :, 2] / 255.0  # red channel
    return heatmap

cnn_scores = []
mobilenet_scores = []

for img_name in os.listdir(CNN_DIR):
    base = img_name.replace("cnn_grad_", "")
    mob_name = f"mobilenet_grad_{base}"

    cnn_path = os.path.join(CNN_DIR, img_name)
    mob_path = os.path.join(MOBILENET_DIR, mob_name)

    if not os.path.exists(mob_path):
        continue

    cnn_heatmap = load_heatmap_from_gradcam(cnn_path)
    mob_heatmap = load_heatmap_from_gradcam(mob_path)

    cnn_scores.append(focus_score(cnn_heatmap))
    mobilenet_scores.append(focus_score(mob_heatmap))

print("CNN Avg Focus:", np.mean(cnn_scores))
print("MobileNet Avg Focus:", np.mean(mobilenet_scores))

print("CNN Std Dev:", np.std(cnn_scores))
print("MobileNet Std Dev:", np.std(mobilenet_scores))


CNN Avg Focus: 0.0003985969387755103
MobileNet Avg Focus: 0.023840082908163262
CNN Std Dev: 0.0006272046639914205
MobileNet Std Dev: 0.023369272817485437
